# Here I wanna experiment with the evaluation

I need an evaluator for the hyperparametertuning on the validationset. (I need a second entrypoint for the final evaluation on the benchmarks (testset) later, or maybe it's even better to use the original benchmark evaluator for comparison)


In [ ]:
# keep this cell (layer 1)
import numpy as np
import torch
from dataclasses import dataclass
from typing import Optional, List

@dataclass
class PredFrame:
    ids: np.ndarray                 # always 1D (N,)
    y_pred: np.ndarray              # (N, D)
    y_true: Optional[np.ndarray]    # (N, D) or None
    eval_mask: Optional[np.ndarray] # (N, D) bool or None

@torch.no_grad()
def infer(model, dataloader, device) -> PredFrame:
    model.eval()
    model.to(device)

    ids_all: List[np.ndarray] = []
    pred_all: List[np.ndarray] = []
    true_all: List[np.ndarray] = []
    mask_all: List[np.ndarray] = []

    row_cursor = 0  # used to generate synthetic IDs when loader provides no IDs

    for batch in dataloader:
        # --- unpack batch ---
        if isinstance(batch, (tuple, list)) and len(batch) == 2:
            # test: (ID, x)
            ID, x = batch
            y = None
            flag = None

        elif isinstance(batch, (tuple, list)) and len(batch) == 3:
            # train/val (current PSML): (x, y, flag)  -> no IDs
            x, y, flag = batch
            B = x.shape[0]
            ID = np.arange(row_cursor, row_cursor + B, dtype=np.int64)
            row_cursor += B

        elif isinstance(batch, (tuple, list)) and len(batch) == 4:
            # train/val (after patch): (ID, x, y, flag)
            ID, x, y, flag = batch

        else:
            raise TypeError(
                f"Unexpected batch format: type={type(batch)} len={len(batch) if hasattr(batch,'__len__') else 'n/a'}"
            )

        # --- IDs -> numpy 1D ---
        #if torch.is_tensor(ID):
        #    ID = ID.detach().cpu().numpy()
        #ids = np.asarray(ID).reshape(-1)  # ensure 1D, avoids zero-dim concat
        ids = ID
        ids_all.append(ids)

        # --- forward ---
        x = x.to(device).float()
        y_hat = model(x).detach().cpu().numpy()
        if y_hat.ndim == 1:
            y_hat = y_hat[:, None]
        pred_all.append(y_hat)

        # --- optional y_true ---
        if y is not None:
            y_np = y.detach().cpu().numpy()
            if y_np.ndim == 1:
                y_np = y_np[:, None]
            true_all.append(y_np)

        # --- optional eval mask ---
        if flag is not None:
            m = flag.detach().cpu().numpy()
            if m.ndim == 1:
                m = m[:, None]
            mask_all.append(m.astype(bool))

    ids = np.concatenate(ids_all, axis=0)
    y_pred = np.concatenate(pred_all, axis=0)
    y_true = np.concatenate(true_all, axis=0) if true_all else None
    eval_mask = np.concatenate(mask_all, axis=0) if mask_all else None

    # sanity checks (optional but helpful)
    assert ids.shape[0] == y_pred.shape[0], "IDs and predictions count mismatch"
    if y_true is not None:
        assert y_true.shape == y_pred.shape, f"y_true {y_true.shape} != y_pred {y_pred.shape}"
    if eval_mask is not None:
        assert eval_mask.shape == y_pred.shape, f"mask {eval_mask.shape} != y_pred {y_pred.shape}"

    return PredFrame(ids=ids, y_pred=y_pred, y_true=y_true, eval_mask=eval_mask)


In [ ]:
from dataloader import TimeSeriesLoader

loader = TimeSeriesLoader(
    task="forecasting",
    root="/media/lll/T9/PSML"
)

train_loader, test_loader = loader.load(
    batch_size=32,
    shuffle=False
)


In [ ]:
from torch.utils.data import DataLoader, Subset
import numpy as np

def small_loader(loader, n_samples=128):
    idx = np.arange(min(n_samples, len(loader.dataset)))
    return DataLoader(
        Subset(loader.dataset, idx),
        batch_size=loader.batch_size,
        shuffle=False
    )

train_small = small_loader(train_loader)
test_small  = small_loader(test_loader)


In [ ]:
from models.mlp import STLFMLP
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = STLFMLP(
    input_dim=1331,      # aus config
    hidden_sizes=[512, 256, 128],
    out_dim=6,
    dropout=0.1,
    use_layernorm= True,
    use_input_norm=True,
    activation="gelu"
)

# optional: Checkpoint
# ckpt = torch.load(".../best-val.pt", map_location=device)
# model.load_state_dict(ckpt["model"])


In [ ]:
preds_train = infer(model, train_small, device)
preds_test  = infer(model, test_small, device)


In [ ]:
# Train / Val
print(preds_train.y_pred.shape)
print(preds_train.y_true.shape if preds_train.y_true is not None else None)

# Test
print(preds_test.y_pred.shape)
print(preds_test.y_true)  # muss None sein


# IDs
print(preds_test.ids[:5])

assert len(np.unique(preds_test.ids)) == len(preds_test.ids)


In [ ]:
## keep this cell
# =============================
# Layer 2: Evaluation
# - val: fast masked MAE/RMSE on your val split (no leakage)
# - test_psml: write PSML prediction CSV(s) + run official PSML evaluator -> summary.json
# =============================

from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import json
import numpy as np
import pandas as pd

# Expects your Layer-1 PredFrame with fields:
#   preds.ids: np.ndarray (N,)
#   preds.y_pred: np.ndarray (N, D)
#   preds.y_true: Optional[np.ndarray] (N, D) for val/train
#   preds.eval_mask: Optional[np.ndarray] (N, D) bool for val/train
#
# (If your PredFrame uses a different field name than eval_mask, adapt below.)


# -----------------------------
# Val metrics (masked)
# -----------------------------
def _to_jsonable(x):
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, dict):
        return {k: _to_jsonable(v) for k,v in x.items()}
    if isinstance(x, list):
        return [_to_jsonable(v) for v in x]
    return x

def write_json(path: Path, obj: dict, overwrite: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and not overwrite:
        raise FileExistsError(f"{path} exists. Use overwrite=True.")
    path.write_text(json.dumps(obj, indent=2) + "\n")

def _masked_flat(yt: np.ndarray, yp: np.ndarray, mask: np.ndarray | None):
    if mask is None:
        return yt.reshape(-1), yp.reshape(-1)
    m = mask.astype(bool)
    return yt[m], yp[m]

def eval_val_metrics(preds, dim_names: list[str] | None = None) -> dict:
    if preds.y_true is None:
        raise ValueError("Need y_true for val metrics.")
    yt_all = preds.y_true
    yp_all = preds.y_pred
    mask = getattr(preds, "eval_mask", None)
    if mask is None:
        mask = getattr(preds, "flags", None)  # falls du es so nennst


    yt, yp = _masked_flat(yt_all, yp_all, mask)
    out = {
        "split": "val",
        "metrics": {
            "mae": float(np.mean(np.abs(yt - yp))),
            "rmse": float(np.sqrt(np.mean((yt - yp) ** 2))),
            "n_obs": int(yt.size),
        }
    }

    D = yp_all.shape[1]
    if dim_names is None:
        dim_names = [f"dim_{d}" for d in range(D)]
    if len(dim_names) != D:
        raise ValueError("dim_names must match y_pred second dim.")

    per_dim = {}
    for d, name in enumerate(dim_names):
        yt_d = yt_all[:, d]
        yp_d = yp_all[:, d]
        m_d = mask[:, d] if mask is not None else None
        yt_f, yp_f = _masked_flat(yt_d, yp_d, m_d)
        per_dim[name] = {
            "mae": float(np.mean(np.abs(yt_f - yp_f))),
            "rmse": float(np.sqrt(np.mean((yt_f - yp_f) ** 2))),
            "n_obs": int(yt_f.size),
        }
    out["per_dim"] = per_dim
    return out

def eval_val_and_save(preds, eval_dir: Path, dim_names: list[str], overwrite: bool = False) -> dict:
    payload = eval_val_metrics(preds, dim_names=dim_names)
    write_json(eval_dir / "val_metrics.json", payload, overwrite=overwrite)
    return payload

# -----------------------------
# PSML test: write prediction CSV(s)
# -----------------------------

@dataclass(frozen=True)
class PsmlColumnMap:
    """
    Maps your model output dims -> PSML prediction columns (mean/U/L).

    If your model outputs D=6 in this order:
      ["yl_t+60","yl_t+1440","yw_t+5","yw_t+30","ys_t+5","ys_t+30"]

    Then PSML expects columns:
      yl_t+60(mean), yl_t+60(U), yl_t+60(L), ...
    """
    base_cols: List[str]  # length D, in the SAME order as preds.y_pred columns

    def mean_cols(self) -> List[str]:
        return [f"{c}(mean)" for c in self.base_cols]


def write_psml_prediction_csv(
    preds,
    out_csv: Path,
    colmap: PsmlColumnMap,
    interval: str = "degenerate",
    std_per_dim: Optional[np.ndarray] = None,
    interval_multiplier: float = 1.96,  # ~95% if you use std-based interval
    overwrite: bool = False,
) -> Path:
    """
    Writes ONE PSML prediction CSV for a loc_year, e.g. CAISO_zone_1_2018.csv

    interval:
      - "degenerate": U=L=mean (evaluator works; MSIS becomes trivial)
      - "std": U/L = mean +/- interval_multiplier * std_per_dim (len D)

    Requirements:
      - preds.ids must be real PSML IDs (from test loader)
      - preds.y_pred shape (N, D)
      - out_csv.name must match GT file name, e.g. "CAISO_zone_1_2018.csv"
    """
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    if out_csv.exists() and not overwrite:
        raise FileExistsError(f"{out_csv} exists. Use overwrite=True.")

    ids = preds.ids
    y_pred = preds.y_pred

    if ids is None:
        raise ValueError("PSML test writing requires real IDs (preds.ids).")
    if y_pred.ndim != 2:
        raise ValueError(f"Expected y_pred 2D (N,D), got shape {y_pred.shape}")
    if y_pred.shape[1] != len(colmap.base_cols):
        raise ValueError(f"y_pred D={y_pred.shape[1]} != len(base_cols)={len(colmap.base_cols)}")

    mean = y_pred.astype(float)

    if interval == "degenerate":
        U = mean
        L = mean
    elif interval == "std":
        if std_per_dim is None:
            raise ValueError("interval='std' requires std_per_dim.")
        std_per_dim = np.asarray(std_per_dim).reshape(-1)
        if std_per_dim.shape[0] != mean.shape[1]:
            raise ValueError("std_per_dim must have shape (D,).")
        U = mean + interval_multiplier * std_per_dim[None, :]
        L = mean - interval_multiplier * std_per_dim[None, :]
    else:
        raise ValueError("interval must be 'degenerate' or 'std'.")

    df = pd.DataFrame(mean, columns=colmap.mean_cols())
    # add U/L columns in the exact names evaluator expects
    for j, base in enumerate(colmap.base_cols):
        df[f"{base}(U)"] = U[:, j]
        df[f"{base}(L)"] = L[:, j]
    df.insert(0, "ID", ids)

    df.to_csv(out_csv, index=False)
    return out_csv


# -----------------------------
# PSML test: run official evaluator -> summary.json
# -----------------------------

def run_psml_evaluate_and_write_summary(
    *,
    psml_root: Path,
    prediction_dir: Path,
    out_summary_json: Path,
    prediction_interval: float = 0.95,
) -> dict:
    """
    Loads ALL prediction CSVs in prediction_dir into the evaluator input_dict,
    runs the official forecasting evaluator, and writes summary.json.

    prediction_dir must contain files like:
      CAISO_zone_1_2018.csv
      CAISO_zone_2_2018.csv
      ...
    """
    from BenchmarkModel.LoadForecasting.evaluating import run_evaluate_forecasting

    input_dict: Dict[str, Dict[str, np.ndarray]] = {}
    for csv_path in sorted(prediction_dir.glob("*.csv")):
        loc_year = csv_path.stem
        df = pd.read_csv(csv_path)
        input_dict[loc_year] = {col: df[col].to_numpy() for col in df.columns}

    if not input_dict:
        raise FileNotFoundError(f"No prediction CSVs found in {prediction_dir}")

    summary = run_evaluate_forecasting(
        root=str(psml_root),
        input_dict=input_dict,
        prediction_interval=prediction_interval,
    )

    out_summary_json.parent.mkdir(parents=True, exist_ok=True)
    with out_summary_json.open("w") as f:
        summary = _to_jsonable(summary)
        json.dump(summary, f, indent=2)

    return summary


In [ ]:
# nutze den vollen test_loader (nicht Subset)
preds_test_full = infer(model, test_loader, device)

pred_dir = Path("tmp_preds")
write_psml_prediction_csv(preds_test_full, pred_dir/"CAISO_zone_1_2018.csv", colmap, overwrite=True)

summary = run_psml_evaluate_and_write_summary(
    psml_root=Path("/media/lll/T9/PSML"),
    prediction_dir=pred_dir,
    out_summary_json=pred_dir/"summary.json",
)
summary


In [ ]:
#### layer 3
####
from __future__ import annotations
from pathlib import Path
from typing import Any, Dict, Optional, Tuple, List
import json

import numpy as np
import matplotlib.pyplot as plt


# -----------------------------
# Helpers
# -----------------------------

def read_json(path: Path) -> Dict[str, Any]:
    with path.open("r") as f:
        return json.load(f)

def to_jsonable(x: Any) -> Any:
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, dict):
        return {k: to_jsonable(v) for k, v in x.items()}
    if isinstance(x, list):
        return [to_jsonable(v) for v in x]
    return x

def _fmt(x: Any, digits: int = 4) -> str:
    if x is None:
        return "-"
    if isinstance(x, (np.floating, np.integer)):
        x = x.item()
    if isinstance(x, float):
        return f"{x:.{digits}f}"
    return str(x)

def _md_table(header: List[str], rows: List[List[str]]) -> str:
    out = []
    out.append("| " + " | ".join(header) + " |")
    out.append("| " + " | ".join(["---"] * len(header)) + " |")
    for r in rows:
        out.append("| " + " | ".join(r) + " |")
    return "\n".join(out) + "\n"


# -----------------------------
# Parse PSML summary
# -----------------------------

def parse_psml_summary(raw: Dict[str, Any]) -> Dict[int, Dict[str, Dict[str, Any]]]:
    """
    Normalizes to: {year:int -> {horizon -> metrics_dict}}
    """
    out: Dict[int, Dict[str, Dict[str, Any]]] = {}
    for y, yv in raw.items():
        try:
            yi = int(y)
        except Exception:
            continue
        if not isinstance(yv, dict):
            continue
        out[yi] = {}
        for horizon, hv in yv.items():
            if isinstance(hv, dict):
                out[yi][horizon] = hv
    return out


# -----------------------------
# Plotters
# -----------------------------

def plot_psml_mae_by_horizon(psml_summary: Dict[str, Any], out_png: Path, year: Optional[int] = None) -> Optional[Path]:
    s = parse_psml_summary(psml_summary)
    if not s:
        return None

    # pick a year that has horizons
    if year is None:
        year = next((y for y in sorted(s.keys()) if len(s[y]) > 0), None)
    if year is None or len(s.get(year, {})) == 0:
        return None

    horizons = s[year]
    labels = sorted(horizons.keys())
    maes = [horizons[h].get("MAE") for h in labels]
    maes = [m.item() if isinstance(m, (np.floating, np.integer)) else float(m) for m in maes]

    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.figure()
    plt.bar(range(len(labels)), maes)
    plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
    plt.ylabel("MAE")
    plt.title(f"PSML MAE by Horizon ({year})")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()
    return out_png

def plot_val_mae_per_dim(val_metrics: Dict[str, Any], out_png: Path) -> Optional[Path]:
    per_dim = val_metrics.get("per_dim")
    if not isinstance(per_dim, dict) or len(per_dim) == 0:
        return None

    labels = list(per_dim.keys())
    maes = [per_dim[k].get("mae") for k in labels]
    maes = [m.item() if isinstance(m, (np.floating, np.integer)) else float(m) for m in maes]

    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.figure()
    plt.bar(range(len(labels)), maes)
    plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
    plt.ylabel("MAE")
    plt.title("Validation MAE per dimension")
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()
    return out_png


# -----------------------------
# Report builder
# -----------------------------

def make_report(eval_dir: Path, *, plot_year: Optional[int] = None) -> Path:
    eval_dir = Path(eval_dir)
    reports_dir = eval_dir / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)

    val_path = eval_dir / "val_metrics.json"
    psml_path = eval_dir / "psml_summary" / "summary.json"

    parts: List[str] = []
    parts.append("# Evaluation Report\n")
    parts.append(f"**eval_dir:** `{eval_dir.as_posix()}`\n")

    # --- VAL ---
    if val_path.exists():
        val = read_json(val_path)
        val = to_jsonable(val)
        m = val.get("metrics", val)

        parts.append("## Validation\n")
        rows = [
            ["MAE", _fmt(m.get("mae"))],
            ["RMSE", _fmt(m.get("rmse"))],
            ["n_obs", _fmt(m.get("n_obs"), digits=0)],
        ]
        parts.append(_md_table(["Metric", "Value"], rows))

        # per-dim table
        per_dim = val.get("per_dim", {})
        if isinstance(per_dim, dict) and len(per_dim) > 0:
            rows = []
            for k in per_dim.keys():
                rows.append([k, _fmt(per_dim[k].get("mae")), _fmt(per_dim[k].get("rmse")), _fmt(per_dim[k].get("n_obs"), digits=0)])
            parts.append("### Per dimension\n")
            parts.append(_md_table(["Dim", "MAE", "RMSE", "n_obs"], rows))

            plot_path = plot_val_mae_per_dim(val, reports_dir / "val_mae_per_dim.png")
            if plot_path:
                parts.append(f"![Validation MAE per dim]({plot_path.name})\n")
    else:
        parts.append("## Validation\n\n_Not found: `val_metrics.json`._\n")

    # --- TEST / PSML ---
    if psml_path.exists():
        raw = read_json(psml_path)
        raw = to_jsonable(raw)
        s = parse_psml_summary(raw)

        parts.append("## PSML Test Summary\n")
        if not s:
            parts.append("_Summary empty/unparseable._\n")
        else:
            for year in sorted(s.keys()):
                parts.append(f"### Year {year}\n")
                horizons = s[year]
                if not horizons:
                    parts.append("_No entries (missing CSVs for this year?)._\n")
                    continue
                rows = []
                for h in sorted(horizons.keys()):
                    hv = horizons[h]
                    rows.append([
                        h,
                        _fmt(hv.get("#locs") or hv.get("locs"), digits=0),
                        _fmt(hv.get("RMSE")),
                        _fmt(hv.get("MAE")),
                        _fmt(hv.get("MAPE")),
                        _fmt(hv.get("MSIS")),
                    ])
                parts.append(_md_table(["Horizon", "#locs", "RMSE", "MAE", "MAPE", "MSIS"], rows))

            plot_path = plot_psml_mae_by_horizon(raw, reports_dir / "psml_mae_by_horizon.png", year=plot_year)
            if plot_path:
                parts.append(f"![PSML MAE by horizon]({plot_path.name})\n")
    else:
        parts.append("## PSML Test Summary\n\n_Not found: `psml_summary/summary.json`._\n")

    report_md = reports_dir / "report.md"
    report_md.write_text("\n".join(parts), encoding="utf-8")
    return report_md


In [ ]:
from pathlib import Path
report = make_report(Path("tmp_eval/val"))     # oder tmp_eval/test
print(report)
print(report.read_text()[:500])


In [ ]:
from pathlib import Path

# 1) Eval-Dir (für Notebook ruhig tmp)
val_dir = Path("tmp_eval/val")
val_dir.mkdir(parents=True, exist_ok=True)

# 2) Dim-Namen passend zu deinem Output
dim_names = ["yl_t+60","yl_t+1440","yw_t+5","yw_t+30","ys_t+5","ys_t+30"]

# 3) Falls dein PredFrame statt eval_mask 'flags' hat, mappen wir kurz:
if getattr(preds_train, "eval_mask", None) is None and getattr(preds_train, "flags", None) is not None:
    preds_train.eval_mask = preds_train.flags.astype(bool)  # nur fürs Notebook

# 4) Val evaluieren + speichern
val_payload = eval_val_and_save(preds_train, val_dir, dim_names=dim_names, overwrite=True)
val_payload
